# 02 — Model Training (Phase 2)
**Project:** From Prediction to Referral — Uncertainty-Aware Pneumonia Detection under Cross-Population Distribution Shift

Trains **EfficientNet-B4** and **ViT-B/16** (ImageNet-pretrained) on the Phase 1–audited Mooney development split and
selects the operating threshold on **Mooney validation only**.

**Data firewall (enforced in code):** this notebook reads only `final_split ∈ {train, val}`. The Mooney ID test (`test_id`)
and every NIH image are never loaded. Split fingerprints must match the Phase 1 audit or the notebook stops.

**Locked design choices**
| Item | Choice | Reason |
|---|---|---|
| Labels | NORMAL = 0, PNEUMONIA = 1 (numeric `y`) | protocol |
| Preprocessing | grayscale → pad to square (black) → resize → 3-channel, ImageNet normalisation | preserves anatomy/aspect ratio; NIH (1024²) gets identical treatment later |
| Resolution | EfficientNet-B4 384², ViT-B/16 224² | native pretrained resolutions |
| Augmentation | affine ±7°, translate ≤5 %, scale 0.95–1.05, brightness/contrast ±10 % | clinically plausible only — **no flips**, no crops |
| Loss | unweighted `BCEWithLogitsLoss` | class weighting distorts probabilities, and calibration is a study endpoint; class imbalance is handled by the validation-derived threshold |
| Head | Dropout(0.3) → Linear(F,256) → GELU → Dropout(0.3) → Linear(256,1) | the two head `nn.Dropout` layers are the MC-Dropout layers used in Notebook 04 |
| Optimiser | AdamW, cosine schedule, 1-epoch warm-up, grad-clip 1.0, AMP fp16 | |
| Model selection / early stopping | lowest validation BCE loss (patience 6, max 30 epochs) | proper scoring rule; AUROC saturates on a 738-image validation set |
| Threshold | primary = maximum validation specificity subject to sensitivity ≥ 0.95 (ties → higher sensitivity); also report 0.50 and Youden | sensitivity-oriented operating point, fixed before any test/NIH inference |

**Kaggle settings:** Accelerator **GPU T4 ×1 or P100**, **Internet ON** (to download ImageNet weights).
Inputs: `paultimothymooney/chest-xray-pneumonia` and your `cxr-ood-notebook01-output` dataset.

In [ ]:
# ------------------------------------------------------------------ CONFIG (edit only this cell)
from pathlib import Path
N01_DIR  = Path("/kaggle/input/datasets/ayubpstyo/cxr-ood-notebook01-output")   # holds mooney_manifest_final.csv
PROJECT  = Path("/kaggle/working/CXR_OOD_Project")
RUN_MODELS = ["efficientnet_b4", "vit_b16"]   # run one per session if time is short, e.g. ["vit_b16"]
SEED = 42
SENS_TARGET = 0.95
SMOKE_TEST = False          # True = 2 epochs on a small subset, only to check the pipeline end-to-end
PRETRAINED = True

MODEL_CFG = {
    "efficientnet_b4": dict(timm_name="efficientnet_b4.ra2_in1k", img_size=384, batch_size=16, accum=2,
                            lr_backbone=1e-4, lr_head=1e-3, weight_decay=1e-2, drop_path=0.1),
    "vit_b16":         dict(timm_name="vit_base_patch16_224.augreg_in21k_ft_in1k", img_size=224, batch_size=32, accum=1,
                            lr_backbone=3e-5, lr_head=1e-3, weight_decay=5e-2, drop_path=0.1),
}
COMMON = dict(max_epochs=30, min_epochs=5, patience=6, warmup_epochs=1, grad_clip=1.0, head_dropout=0.3, head_hidden=256)

# Phase 1 locked split fingerprints (sha256 of sorted filepaths) — notebook stops if they differ
EXPECTED = {
    "train": dict(n=4468, pos=3315, neg=1153, fp="762c050910f4560893544f55ef702ba22d80dd948d0a9b007a57b9f49ce79f4b"),
    "val":   dict(n=738,  pos=543,  neg=195,  fp="13d0556c2de25782967c5cd20ceb54508ca4b8061cdca9e0c9c6ae0bff2f139c"),
}

## Shared module — written once and reused unchanged by Notebooks 03–05 (same preprocessing, same model class)

In [ ]:
COMMON_CODE = r'''
"""cxr_common.py — preprocessing and model definition shared by Notebooks 02-05. Do not edit after Phase 2."""
import numpy as np, torch, torch.nn as nn, timm
from PIL import Image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def load_letterboxed(path, size):
    """Grayscale -> pad to square with black -> resize (Lanczos). Returns uint8 array (size, size)."""
    im = Image.open(path).convert("L")
    w, h = im.size; s = max(w, h)
    canvas = Image.new("L", (s, s), 0)
    canvas.paste(im, ((s - w) // 2, (s - h) // 2))
    return np.asarray(canvas.resize((size, size), Image.LANCZOS), dtype=np.uint8)

def to_model_tensor(x_uint8):
    """(B,1,H,W) or (1,H,W) uint8 tensor -> float, 3-channel, ImageNet-normalised."""
    x = x_uint8.float().div(255.0)
    x = x.repeat_interleave(3, dim=-3)
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=x.device).view(3, 1, 1)
    return (x - mean) / std

class CXRClassifier(nn.Module):
    """timm backbone (num_classes=0) + dropout head. head_dropout1/2 are the MC-Dropout layers."""
    def __init__(self, timm_name, pretrained=True, drop_path=0.1, head_dropout=0.3, head_hidden=256):
        super().__init__()
        self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path)
        f = self.backbone.num_features
        self.head_dropout1 = nn.Dropout(head_dropout)
        self.fc1 = nn.Linear(f, head_hidden)
        self.act = nn.GELU()
        self.head_dropout2 = nn.Dropout(head_dropout)
        self.fc2 = nn.Linear(head_hidden, 1)
    def features(self, x):
        return self.backbone(x)
    def head(self, f):
        return self.fc2(self.head_dropout2(self.act(self.fc1(self.head_dropout1(f))))).squeeze(-1)
    def forward(self, x):
        return self.head(self.features(x))
    def mc_dropout_modules(self):
        return [self.head_dropout1, self.head_dropout2]
'''
CODE_DIR = PROJECT / "code"; CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "cxr_common.py").write_text(COMMON_CODE)
import sys; sys.path.insert(0, str(CODE_DIR))
import importlib, cxr_common; importlib.reload(cxr_common)
from cxr_common import load_letterboxed, to_model_tensor, CXRClassifier

In [ ]:
import os, json, time, math, random, hashlib, platform, subprocess, shutil, datetime
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np, pandas as pd
import torch, torch.nn as nn, timm, torchvision
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from concurrent.futures import ThreadPoolExecutor
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

DIRS = {k: PROJECT / k for k in ["checkpoints", "predictions", "metrics", "figures", "logs", "configs"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    raise SystemExit("No GPU detected — set Accelerator to GPU (T4 or P100) and restart.")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_everything(SEED)

ENV = dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__, timm=timm.__version__,
           numpy=np.__version__, pandas=pd.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
           gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu", platform=platform.platform(),
           timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
(DIRS["logs"] / "environment_02.json").write_text(json.dumps(ENV, indent=2))
(DIRS["logs"] / "pip_freeze_02.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(json.dumps(ENV, indent=2))

## 1. Load the audited development split (train/val only) and verify it against Phase 1

In [ ]:
man = pd.read_csv(N01_DIR / "mooney_manifest_final.csv")
SPLITS = {}
for s in ["train", "val"]:
    d = man[man["final_split"] == s][["filepath", "filename", "patient_id", "y", "label", "original_split", "sha256"]].copy()
    d = d.sort_values("filepath").reset_index(drop=True)
    fp = hashlib.sha256("\n".join(d.filepath).encode()).hexdigest()
    e = EXPECTED[s]
    ok = (len(d), int(d.y.sum()), int((d.y == 0).sum()), fp) == (e["n"], e["pos"], e["neg"], e["fp"])
    print(f"{s}: n={len(d)} pos={int(d.y.sum())} neg={int((d.y==0).sum())} fingerprint_ok={fp == e['fp']}")
    if not ok:
        raise SystemExit(f"{s} split does not match the Phase 1 audit — stop and report.")
    SPLITS[s] = d
assert not (set(SPLITS["train"].patient_id) & set(SPLITS["val"].patient_id)), "train/val patient overlap"
del man   # test_id and NIH rows are never kept in memory
if SMOKE_TEST:
    SPLITS = {s: pd.concat([g.sample(min(len(g), 24), random_state=SEED) for _, g in d.groupby("y")]).sort_values("filepath").reset_index(drop=True)
              for s, d in SPLITS.items()}
    COMMON.update(max_epochs=2, min_epochs=1, patience=1)

## 2. Cache letterboxed images in RAM (one decode per image, per resolution)

In [ ]:
CACHE = {}
def get_cache(size):
    if size in CACHE: return CACHE[size]
    out = {}
    for s, d in SPLITS.items():
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
            arrs = list(ex.map(lambda p: load_letterboxed(p, size), d.filepath))
        out[s] = np.stack(arrs)
        print(f"cached {s} @ {size}px: {out[s].shape} in {time.time()-t0:.0f}s")
    CACHE[size] = out
    return out

class CXRDataset(Dataset):
    def __init__(self, imgs, labels, train):
        self.imgs, self.labels, self.train = imgs, labels.astype(np.float32), train
        self.aug = T.Compose([
            T.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05), fill=0),
            T.ColorJitter(brightness=0.1, contrast=0.1),
        ])
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        x = torch.from_numpy(self.imgs[i]).unsqueeze(0)          # (1,H,W) uint8
        if self.train: x = self.aug(x)
        return x, torch.tensor(self.labels[i])

def seed_worker(wid):
    s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)

## 3. Metrics and threshold selection (validation only)

In [ ]:
def metrics_at(y, p, thr):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    sd = lambda a, b: float(a / b) if b else float("nan")
    sens, spec, ppv, npv = sd(tp, tp + fn), sd(tn, tn + fp), sd(tp, tp + fp), sd(tn, tn + fn)
    return dict(threshold=float(thr), sensitivity=sens, specificity=spec, ppv=ppv, npv=npv,
                f1=sd(2 * ppv * sens, ppv + sens) if not math.isnan(ppv) else float("nan"),
                accuracy=sd(tp + tn, len(y)), tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

def select_thresholds(y, p, sens_target):
    cand = np.unique(np.concatenate([p, [0.5]]))
    rows = [metrics_at(y, p, t) for t in cand]
    df = pd.DataFrame(rows)
    ok = df[df.sensitivity >= sens_target]
    # among thresholds meeting the sensitivity target: max specificity; ties -> higher sensitivity; then higher threshold
    t_sens = float(ok.sort_values(["specificity", "sensitivity", "threshold"], ascending=False).iloc[0].threshold)
    youden = df.assign(j=df.sensitivity + df.specificity - 1).sort_values(["j", "threshold"], ascending=[False, False]).iloc[0]
    return dict(default_0_50=metrics_at(y, p, 0.5),
                sensitivity_target=dict(target=sens_target, **metrics_at(y, p, t_sens)),
                youden=metrics_at(y, p, float(youden.threshold)))

def logit(p): p = np.clip(p, 1e-7, 1 - 1e-7); return float(np.log(p / (1 - p)))

## 4. Training

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); L, Y = [], []
    for x, y in loader:
        x = to_model_tensor(x.to(DEVICE, non_blocking=True))
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            L.append(model(x).float().cpu())
        Y.append(y)
    return torch.cat(L).numpy(), torch.cat(Y).numpy()

def train_one(name):
    cfg = {**MODEL_CFG[name], **COMMON}
    seed_everything(SEED)
    cache = get_cache(cfg["img_size"])
    g = torch.Generator(); g.manual_seed(SEED)
    nw = min(4, os.cpu_count())
    dl_tr = DataLoader(CXRDataset(cache["train"], SPLITS["train"].y.values, True), batch_size=cfg["batch_size"], shuffle=True,
                       num_workers=nw, worker_init_fn=seed_worker, generator=g, pin_memory=True, drop_last=True, persistent_workers=nw > 0)
    dl_va = DataLoader(CXRDataset(cache["val"], SPLITS["val"].y.values, False), batch_size=cfg["batch_size"] * 2, shuffle=False,
                       num_workers=nw, pin_memory=True)
    model = CXRClassifier(cfg["timm_name"], pretrained=PRETRAINED, drop_path=cfg["drop_path"],
                          head_dropout=cfg["head_dropout"], head_hidden=cfg["head_hidden"]).to(DEVICE)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": cfg["lr_backbone"]},
                             {"params": head_params, "lr": cfg["lr_head"]}], weight_decay=cfg["weight_decay"])
    steps_per_epoch = max(1, len(dl_tr) // cfg["accum"])
    total, warm = steps_per_epoch * cfg["max_epochs"], steps_per_epoch * cfg["warmup_epochs"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    crit = nn.BCEWithLogitsLoss()
    ckpt = DIRS["checkpoints"] / f"{name}_best.pt"
    hist, best, bad = [], float("inf"), 0
    for ep in range(1, cfg["max_epochs"] + 1):
        model.train(); t0 = time.time(); run, n = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, (x, y) in enumerate(dl_tr):
            x = to_model_tensor(x.to(DEVICE, non_blocking=True)); y = y.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = crit(model(x).float(), y) / cfg["accum"]
            scaler.scale(loss).backward()
            if (i + 1) % cfg["accum"] == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            run += loss.item() * cfg["accum"] * len(y); n += len(y)
        lv, yv = predict(model, dl_va)
        pv = 1 / (1 + np.exp(-lv))
        vloss = float(crit(torch.tensor(lv), torch.tensor(yv)).item())
        m05 = metrics_at(yv, pv, 0.5)
        row = dict(epoch=ep, train_loss=run / n, val_loss=vloss, val_auroc=roc_auc_score(yv, pv),
                   val_auprc=average_precision_score(yv, pv), val_sens_050=m05["sensitivity"], val_spec_050=m05["specificity"],
                   lr_backbone=opt.param_groups[0]["lr"], seconds=time.time() - t0)
        hist.append(row)
        improved = vloss < best - 1e-4
        if improved:
            best, bad = vloss, 0
            torch.save({"model_state": model.state_dict(), "model_name": name, "config": cfg, "epoch": ep,
                        "val_loss": vloss, "seed": SEED, "env": ENV}, ckpt)
        else:
            bad += 1
        print(f"[{name}] ep {ep:02d} train {row['train_loss']:.4f} | val loss {vloss:.4f} AUROC {row['val_auroc']:.4f} "
              f"AUPRC {row['val_auprc']:.4f} sens {m05['sensitivity']:.3f} spec {m05['specificity']:.3f} | "
              f"{row['seconds']:.0f}s {'*' if improved else ''}")
        if ep >= cfg["min_epochs"] and bad >= cfg["patience"]:
            print(f"[{name}] early stop at epoch {ep}; best val loss {best:.4f}"); break
    pd.DataFrame(hist).to_csv(DIRS["logs"] / f"{name}_history.csv", index=False)

    # ---- best checkpoint → validation predictions → thresholds
    state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state["model_state"])
    lv, yv = predict(model, dl_va); pv = 1 / (1 + np.exp(-lv))
    vp = SPLITS["val"][["filename", "filepath", "patient_id", "y"]].copy()
    vp.insert(0, "cohort", "MOONEY_VAL"); vp.insert(0, "dataset", "MOONEY_KERMANY"); vp.insert(0, "model", name)
    vp["logit"], vp["prob"] = lv, pv
    thr = select_thresholds(yv, pv, SENS_TARGET)
    for k in ["default_0_50", "sensitivity_target", "youden"]:
        thr[k]["threshold_logit"] = logit(thr[k]["threshold"])
    vp["pred_050"] = (pv >= 0.5).astype(int)
    vp["pred_sens_target"] = (pv >= thr["sensitivity_target"]["threshold"]).astype(int)
    vp.to_csv(DIRS["predictions"] / f"{name}_val_predictions.csv", index=False)
    summary = dict(model=name, timm_name=cfg["timm_name"], best_epoch=state["epoch"], epochs_run=len(hist),
                   best_val_loss=state["val_loss"], val_auroc=roc_auc_score(yv, pv), val_auprc=average_precision_score(yv, pv),
                   n_val=len(yv), val_pos=int(yv.sum()), thresholds=thr,
                   primary_operating_threshold=thr["sensitivity_target"]["threshold"],
                   primary_operating_threshold_logit=thr["sensitivity_target"]["threshold_logit"],
                   note="Threshold fixed on Mooney validation; applied unchanged to ID test and NIH. "
                        "After temperature scaling (Notebook 03) the equivalent rule is logit/T >= threshold_logit/T.")
    (DIRS["metrics"] / f"{name}_val_metrics.json").write_text(json.dumps(summary, indent=2))
    (DIRS["configs"] / f"{name}_train_config.json").write_text(json.dumps(dict(model=name, seed=SEED, sens_target=SENS_TARGET,
        pretrained=PRETRAINED, smoke_test=SMOKE_TEST, split_fingerprints={k: v["fp"] for k, v in EXPECTED.items()},
        preprocessing="grayscale; pad-to-square black; Lanczos resize; 3-channel; ImageNet mean/std",
        augmentation="RandomAffine(deg=7, translate=0.05, scale=0.95-1.05, fill=0); ColorJitter(b=0.1, c=0.1); no flips/crops",
        loss="BCEWithLogitsLoss (unweighted)", early_stopping="val BCE loss", **cfg), indent=2))
    plot_curves(name, pd.DataFrame(hist), state["epoch"])
    print(json.dumps({k: summary[k] for k in ["best_epoch", "best_val_loss", "val_auroc", "val_auprc", "primary_operating_threshold"]}, indent=2))
    del model; torch.cuda.empty_cache()
    return summary

def plot_curves(name, h, best_ep):
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
    fig, ax = plt.subplots(1, 3, figsize=(10, 3))
    ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="validation")
    ax[0].set(xlabel="Epoch", ylabel="BCE loss", title="Loss"); ax[0].legend(frameon=False)
    ax[1].plot(h.epoch, h.val_auroc, label="AUROC"); ax[1].plot(h.epoch, h.val_auprc, label="AUPRC")
    ax[1].set(xlabel="Epoch", ylabel="Validation score", title="Discrimination"); ax[1].legend(frameon=False)
    ax[2].plot(h.epoch, h.lr_backbone); ax[2].set(xlabel="Epoch", ylabel="Backbone LR", title="Learning rate")
    for a in ax: a.axvline(best_ep, ls="--", lw=0.8, color="grey")
    fig.suptitle(f"{name} — training curves (dashed: selected epoch)", fontsize=10); fig.tight_layout()
    for ext in ["png", "pdf"]: fig.savefig(DIRS["figures"] / f"{name}_training_curves.{ext}", dpi=300)
    plt.close(fig)

In [ ]:
SUMMARIES = {}
for name in RUN_MODELS:
    SUMMARIES[name] = train_one(name)

## 5. Threshold report (validation only) and output ZIP

In [ ]:
rows = []
for name, s in SUMMARIES.items():
    for k, lab in [("default_0_50", "0.50 (default)"), ("sensitivity_target", f"sens ≥ {SENS_TARGET:.2f} (primary)"), ("youden", "Youden (secondary)")]:
        t = s["thresholds"][k]
        rows.append(dict(model=name, operating_point=lab, threshold=round(t["threshold"], 6), threshold_logit=round(t["threshold_logit"], 4), sensitivity=t["sensitivity"],
                         specificity=t["specificity"], ppv=t["ppv"], npv=t["npv"], f1=t["f1"], accuracy=t["accuracy"],
                         tp=t["tp"], fp=t["fp"], tn=t["tn"], fn=t["fn"]))
thr_table = pd.DataFrame(rows)
thr_table.to_csv(DIRS["metrics"] / "validation_threshold_table.csv", index=False)
print(thr_table.round(4).to_string(index=False))
print("\nValidation AUROC/AUPRC:", {n: (round(s["val_auroc"], 4), round(s["val_auprc"], 4)) for n, s in SUMMARIES.items()})

zip_base = PROJECT / "phase2_training_outputs"
tmp = PROJECT / "_zip02"; shutil.rmtree(tmp, ignore_errors=True)
for k in ["checkpoints", "predictions", "metrics", "figures", "logs", "configs", "code"]:
    shutil.copytree(PROJECT / k, tmp / k, dirs_exist_ok=True)
shutil.make_archive(str(zip_base), "zip", tmp); shutil.rmtree(tmp)
print("ZIP →", f"{zip_base}.zip", f"({os.path.getsize(f'{zip_base}.zip')/1e6:.0f} MB)")
print("\nPhase 2 complete. Do NOT run Notebook 03 until these validation results are reviewed.")